# Marketplace Growth & Customer Experience Analysis

## C1. Raw Data Audit

### Objective

This notebook performs a structured audit of the raw Olist Marketplace datasets before any data cleaning, transformation, database design, or SQL loading is performed.

The audit is intended to:

- Understand the structure and grain of each source table.
- Validate primary and candidate keys.
- Assess missing values and duplicate records.
- Validate relationships between tables.
- Identify potential data type, categorical, numerical, temporal, and text-quality issues.
- Document findings that require further investigation during data cleaning.

No source values are modified in this notebook. Potential issues are identified and carried forward to `C2_data_cleaning.ipynb` for investigation and resolution.

## Audit Scope

The notebook is organised into the following stages:

1. Import libraries
2. Load source datasets
3. Build the dataset inventory
4. Define assumptions requiring validation
5. Audit dataset structure
6. Validate table grain and keys
7. Assess duplicate records
8. Assess missing values
9. Review data types
10. Validate referential integrity and cardinality
11. Assess categorical values
12. Assess numerical values
13. Assess dates and timestamps
14. Assess text fields and lookup coverage
15. Summarise findings for the cleaning stage

## 1. Import Libraries

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# File handling
from pathlib import Path

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

## 2. Load Source Datasets

In [2]:
DATA_PATH = Path("Data")

# Load Source Datasets
orders = pd.read_csv(DATA_PATH / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")
customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")
sellers = pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv")
payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")
geolocation = pd.read_csv(DATA_PATH / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(DATA_PATH / "product_category_name_translation.csv")


## 3. Dataset Inventory

The inventory records the row count, column count, and memory usage of each source table. These values establish a baseline for detecting unexpected record loss or expansion during cleaning and transformation.

In [3]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation,
}

inventory = pd.DataFrame(
    [ {
            "table_name": table_name,
            "rows": len(dataframe),
            "columns": dataframe.shape[1],
            "memory_mb": round(dataframe.memory_usage(index=True, deep=True,).sum() / (1024 ** 2),2,),}
             for table_name, dataframe in datasets.items()])

display(inventory)

,table_name,rows,columns,memory_mb
0,customers,99441,5,26.59
1,orders,99441,8,52.94
2,order_items,112650,7,35.99
3,payments,103886,5,16.23
4,reviews,99224,7,39.12
5,products,32951,9,6.30
6,sellers,3095,4,0.59
7,geolocation,1000163,5,129.38
8,category_translation,71,2,0.01


### Audit Interpretation

This audit verifies the successful loading of all nine source tables. It establishes the baseline row counts required to detect data loss during the subsequent cleaning and transformation phases.

## 4. Assumptions Requiring Validation

Prior to data analysis, the following structural assumptions must be empirically verified rather than inferred solely from system documentation.

| Table | Assumption |
|---|---|
| customers | `customer_id` uniquely identifies a customer record |
| customers | `customer_unique_id` can occur across multiple customer records |
| orders | `order_id` uniquely identifies an order |
| order_items | `order_id` and `order_item_id` uniquely identify an order item |
| products | `product_id` uniquely identifies a product |
| sellers | `seller_id` uniquely identifies a seller |
| payments | `order_id` and `payment_sequential` uniquely identify a payment record |
| reviews | `review_id` may not be unique; a composite key may be required |
| category_translation | `product_category_name` uniquely identifies a translation |
| geolocation | ZIP-code prefixes may have multiple coordinate records |

Relationships between tables must also be validated for orphan records and observed cardinality.

## 5. Dataset Structure Audit

For every source table, the structure audit records:

- Column names
- Current pandas data types
- Non-null counts
- Missing counts
- Number of distinct values

This establishes the raw schema before any type conversions or cleaning decisions are applied.

In [4]:
def structure_summary(
    dataframe: pd.DataFrame,
    table_name: str,) -> pd.DataFrame:
    """Return a structural summary for one DataFrame."""

    return pd.DataFrame({   "table_name": table_name,
                            "column_name": dataframe.columns,
                            "data_type": dataframe.dtypes.astype(str).values,
                            "non_null_count": dataframe.notna().sum().values,
                            "null_count": dataframe.isna().sum().values,
                            "unique_values": dataframe.nunique(dropna=True).values,})

structure_audit = pd.concat([
        structure_summary(dataframe, table_name)
        for table_name, dataframe in datasets.items()],
        ignore_index=True,)

display(structure_audit)

,table_name,column_name,data_type,non_null_count,null_count,unique_values
0,customers,customer_id,object,99441,0,99441
1,customers,customer_unique_id,object,99441,0,96096
2,customers,customer_zip_code_prefix,int64,99441,0,14994
3,customers,customer_city,object,99441,0,4119
4,customers,customer_state,object,99441,0,27
5,orders,order_id,object,99441,0,99441
6,orders,customer_id,object,99441,0,99441
7,orders,order_status,object,99441,0,8
8,orders,order_purchase_timestamp,object,99441,0,98875
9,orders,order_approved_at,object,99281,160,90733


## 6. Table Grain and Key Validation

Table grain defines what one row represents. Key validation determines whether the proposed identifiers uniquely represent that documented grain.

A proposed key is considered valid when:

- Its required columns contain no missing values.
- The key combination is unique across all rows.

In [5]:
def validate_key(
    dataframe: pd.DataFrame,
    table_name: str,
    key_columns: list[str],
) -> dict:
    """Validate missingness and uniqueness of a proposed key."""

    missing_key_rows = dataframe[key_columns].isna().any(axis=1).sum()
    unique_combinations = dataframe[key_columns].drop_duplicates().shape[0]
    total_rows = len(dataframe)

    return {
        "table_name": table_name,
        "key_columns": " + ".join(key_columns),
        "total_rows": total_rows,
        "unique_combinations": unique_combinations,
        "missing_key_rows": int(missing_key_rows),
        "is_unique": unique_combinations == total_rows,
        "is_complete": missing_key_rows == 0,
        "valid_key": (
            unique_combinations == total_rows
            and missing_key_rows == 0),}

proposed_keys = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "category_translation": ["product_category_name"],
}

key_validation = pd.DataFrame(
           [validate_key(
            datasets[table_name],
            table_name,
            key_columns,)
        for table_name, key_columns in proposed_keys.items()])

display(key_validation)

,table_name,key_columns,total_rows,unique_combinations,missing_key_rows,is_unique,is_complete,valid_key
0,customers,customer_id,99441,99441,0,True,True,True
1,orders,order_id,99441,99441,0,True,True,True
2,order_items,order_id + order_item_id,112650,112650,0,True,True,True
3,products,product_id,32951,32951,0,True,True,True
4,sellers,seller_id,3095,3095,0,True,True,True
5,payments,order_id + payment_sequential,103886,103886,0,True,True,True
6,reviews,review_id + order_id,99224,99224,0,True,True,True
7,category_translation,product_category_name,71,71,0,True,True,True


In [6]:
orders_per_unique_customer = (
    customers
    .groupby("customer_unique_id")
    .size()
    .rename("customer_record_count")
)

display(
    orders_per_unique_customer
    .value_counts()
    .sort_index()
    .rename("number_of_customers")
    .to_frame()
)

,number_of_customers
customer_record_count,
1,93099
2,2745
3,203
4,30
5,8
6,6
7,3
9,1
17,1


### Audit Interpretation

- `customer_id` identifies a customer record and is unique.
- `customer_unique_id` represents the underlying customer across orders and is not unique in the customers table.
- Composite keys must be evaluated as combinations rather than by validating each component independently.
- The geolocation table is not assigned a primary key because the source data can contain multiple records for the same ZIP-code prefix and coordinate area.

## 7. Duplicate Record Analysis

This section distinguishes between:

- Exact duplicate rows
- Duplicate values in proposed key columns
- Repeated business identifiers that may be valid

A repeated identifier is not automatically a duplicate record. Its validity depends on the table grain.

In [7]:
duplicate_summary_rows = []

for table_name, dataframe in datasets.items():
    duplicate_mask = dataframe.duplicated(keep=False)
    duplicate_rows = dataframe.loc[duplicate_mask]

    duplicate_group_count = (
        duplicate_rows
        .drop_duplicates()
        .shape[0]
    )

    duplicate_summary_rows.append(
           {"table_name": table_name,
            "total_rows": len(dataframe),
            "rows_in_duplicate_groups": int(duplicate_mask.sum()),
            "duplicate_groups": int(duplicate_group_count),
            "excess_duplicate_rows": int(
                dataframe.duplicated(keep="first").sum()),
            "duplicate_row_pct": round(duplicate_mask.mean() * 100,2,),
        }
    )

duplicate_summary = pd.DataFrame(duplicate_summary_rows)

display(duplicate_summary)

,table_name,total_rows,rows_in_duplicate_groups,duplicate_groups,excess_duplicate_rows,duplicate_row_pct
0,customers,99441,0,0,0,0.00
1,orders,99441,0,0,0,0.00
2,order_items,112650,0,0,0,0.00
3,payments,103886,0,0,0,0.00
4,reviews,99224,0,0,0,0.00
5,products,32951,0,0,0,0.00
6,sellers,3095,0,0,0,0.00
7,geolocation,1000163,390005,128174,261831,38.99
8,category_translation,71,0,0,0,0.00


In [8]:
geolocation_exact_duplicates = geolocation.loc[geolocation.duplicated(keep=False)].sort_values(list(geolocation.columns))

display(geolocation_exact_duplicates.head(20))

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
519,1001,-23.55,-46.63,sao paulo,SP
583,1001,-23.55,-46.63,sao paulo,SP
818,1001,-23.55,-46.63,sao paulo,SP
206,1001,-23.55,-46.63,sao paulo,SP
429,1001,-23.55,-46.63,sao paulo,SP
596,1001,-23.55,-46.63,sao paulo,SP
639,1001,-23.55,-46.63,sao paulo,SP
771,1001,-23.55,-46.63,sao paulo,SP
912,1001,-23.55,-46.63,sao paulo,SP
985,1001,-23.55,-46.63,sao paulo,SP


In [9]:
duplicate_review_ids = reviews.loc[reviews["review_id"].duplicated(keep=False)].sort_values(["review_id", "order_id"])

display(duplicate_review_ids.head(20))

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


### Interpretation Guidance

- Exact duplicate rows require investigation before removal.
- Repeated geolocation ZIP-code prefixes are expected because multiple coordinates may describe the same postal area.
- A duplicated `review_id` does not automatically indicate a duplicate review because the validated review grain uses the combination of `review_id` and `order_id`.
- No records should be deleted during the audit stage.

## 8. Missing Value Analysis

This section quantifies missing values by table and column. The missingness patterns documented here serve strictly as empirical observations. Determining whether missing entries are structurally expected, problematic, or require imputation will be addressed during the data cleaning phase.

In [10]:
missing_value_summary = pd.concat(
    [pd.DataFrame({
                "table_name": table_name,
                "column_name": dataframe.columns,
                "total_rows": len(dataframe),
                "missing_count": dataframe.isna().sum().values,
                "missing_pct": (
                    dataframe.isna().mean().values * 100
                ).round(2),
                "data_type": dataframe.dtypes.astype(str).values,
            })
        for table_name, dataframe in datasets.items()
    ],ignore_index=True,)

missing_value_summary = (
    missing_value_summary
    .query("missing_count > 0")
    .sort_values(
        ["missing_pct", "table_name"],
        ascending=[False, True],).reset_index(drop=True))

display(missing_value_summary)

,table_name,column_name,total_rows,missing_count,missing_pct,data_type
0,reviews,review_comment_title,99224,87656,88.34,object
1,reviews,review_comment_message,99224,58247,58.70,object
2,orders,order_delivered_customer_date,99441,2965,2.98,object
3,products,product_category_name,32951,610,1.85,object
4,products,product_name_lenght,32951,610,1.85,float64
5,products,product_description_lenght,32951,610,1.85,float64
6,products,product_photos_qty,32951,610,1.85,float64
7,orders,order_delivered_carrier_date,99441,1783,1.79,object
8,orders,order_approved_at,99441,160,0.16,object
9,products,product_weight_g,32951,2,0.01,float64


In [11]:
missing_by_table = (
    missing_value_summary
    .groupby("table_name", as_index=False)
    .agg(
        columns_with_missing=("column_name", "count"),
        total_missing_values=("missing_count", "sum"),
    )
    .sort_values("total_missing_values", ascending=False)
)

display(missing_by_table)

,table_name,columns_with_missing,total_missing_values
2,reviews,2,145903
0,orders,3,4908
1,products,8,2448


### Preliminary Missing Value Classification

Missing values should later be classified as one of the following:

- Expected because of the business process
- Optional descriptive information
- Conditionally applicable
- Unexplained and requiring investigation
- Structurally invalid

## 9. Data Type Assessment

This section focuses on datetime fields and identifier columns that require additional interpretation before database loading.

Datetime columns are tested for parseability and conversion requirements. Identifier columns are reviewed for their source data types, missing values, and numbers of distinct values.

No source columns are converted in this notebook.

In [12]:
datetime_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": [
        "shipping_limit_date",
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}

In [13]:
datetime_type_audit = []

for table_name, columns in datetime_columns.items():
    dataframe = datasets[table_name]

    for column in columns:
        parsed_values = pd.to_datetime(
            dataframe[column],
            errors="coerce",
        )

        source_non_null = dataframe[column].notna().sum()
        parsed_non_null = parsed_values.notna().sum()

        datetime_type_audit.append(
            {
                "table_name": table_name,
                "column_name": column,
                "current_dtype": str(dataframe[column].dtype),
                "expected_dtype": "datetime",
                "source_non_null": int(source_non_null),
                "successfully_parsed": int(parsed_non_null),
                "parse_failures": int(
                    source_non_null - parsed_non_null
                ),
                "requires_conversion": (not pd.api.types.is_datetime64_any_dtype(dataframe[column])),
            }
        )

datetime_type_audit = pd.DataFrame(datetime_type_audit)

display(datetime_type_audit)

,table_name,column_name,current_dtype,expected_dtype,source_non_null,successfully_parsed,parse_failures,requires_conversion
0,orders,order_purchase_timestamp,object,datetime,99441,99441,0,True
1,orders,order_approved_at,object,datetime,99281,99281,0,True
2,orders,order_delivered_carrier_date,object,datetime,97658,97658,0,True
3,orders,order_delivered_customer_date,object,datetime,96476,96476,0,True
4,orders,order_estimated_delivery_date,object,datetime,99441,99441,0,True
5,order_items,shipping_limit_date,object,datetime,112650,112650,0,True
6,reviews,review_creation_date,object,datetime,99224,99224,0,True
7,reviews,review_answer_timestamp,object,datetime,99224,99224,0,True


In [14]:
identifier_columns = []

for table_name, dataframe in datasets.items():
    for column in dataframe.columns:
        if column.endswith("_id"):
            identifier_columns.append(
                {
                    "table_name": table_name,
                    "column_name": column,
                    "current_dtype": str(dataframe[column].dtype),
                    "null_count": int(dataframe[column].isna().sum()),
                    "unique_values": int(dataframe[column].nunique(dropna=True)),
                }
            )

identifier_type_audit = pd.DataFrame(identifier_columns)

display(identifier_type_audit)

,table_name,column_name,current_dtype,null_count,unique_values
0,customers,customer_id,object,0,99441
1,customers,customer_unique_id,object,0,96096
2,orders,order_id,object,0,99441
3,orders,customer_id,object,0,99441
4,order_items,order_id,object,0,98666
5,order_items,order_item_id,int64,0,21
6,order_items,product_id,object,0,32951
7,order_items,seller_id,object,0,3095
8,payments,order_id,object,0,99440
9,reviews,review_id,object,0,98410


### Note :

Although ZIP-code prefixes appear numeric within source files, they function strictly as spatial identifiers rather than quantitative metrics. Consequently, their final schema representation must be determined during database design and data cleaning phases. This determination should be guided by relational join requirements, formatting preservation (such as leading zeros), and source value integrity, rather than mathematical utility.

## 10. Referential Integrity and Cardinality Validation

This procedure validates whether foreign key references in child tables exist within their corresponding parent tables to ensure referential integrity. Additionally, the audit measures empirical relationship cardinality to verify that actual data patterns align with documented schema expectations.

In [15]:
def orphan_count(
    child_dataframe: pd.DataFrame,
    child_key: str,
    parent_dataframe: pd.DataFrame,
    parent_key: str,
) -> int:
    """Count non-null child keys with no matching parent key."""

    child_values = child_dataframe[child_key].dropna()
    parent_values = parent_dataframe[parent_key].dropna()

    return int((~child_values.isin(parent_values)).sum())

In [16]:
relationship_checks = [
    {
        "relationship": "orders → customers",
        "child_table": "orders",
        "child_key": "customer_id",
        "parent_table": "customers",
        "parent_key": "customer_id",
    },
    {
        "relationship": "order_items → orders",
        "child_table": "order_items",
        "child_key": "order_id",
        "parent_table": "orders",
        "parent_key": "order_id",
    },
    {
        "relationship": "order_items → products",
        "child_table": "order_items",
        "child_key": "product_id",
        "parent_table": "products",
        "parent_key": "product_id",
    },
    {
        "relationship": "order_items → sellers",
        "child_table": "order_items",
        "child_key": "seller_id",
        "parent_table": "sellers",
        "parent_key": "seller_id",
    },
    {
        "relationship": "payments → orders",
        "child_table": "payments",
        "child_key": "order_id",
        "parent_table": "orders",
        "parent_key": "order_id",
    },
    {
        "relationship": "reviews → orders",
        "child_table": "reviews",
        "child_key": "order_id",
        "parent_table": "orders",
        "parent_key": "order_id",
    },
]

In [17]:
referential_integrity = pd.DataFrame(
    [
        {
            **check,
            "orphan_rows": orphan_count(
                datasets[check["child_table"]],
                check["child_key"],
                datasets[check["parent_table"]],
                check["parent_key"],
            ),
        }
        for check in relationship_checks
    ]
)

referential_integrity["status"] = np.where(
    referential_integrity["orphan_rows"].eq(0),
    "Valid",
    "Requires investigation",
)

display(referential_integrity)

,relationship,child_table,child_key,parent_table,parent_key,orphan_rows,status
0,orders → customers,orders,customer_id,customers,customer_id,0,Valid
1,order_items → orders,order_items,order_id,orders,order_id,0,Valid
2,order_items → products,order_items,product_id,products,product_id,0,Valid
3,order_items → sellers,order_items,seller_id,sellers,seller_id,0,Valid
4,payments → orders,payments,order_id,orders,order_id,0,Valid
5,reviews → orders,reviews,order_id,orders,order_id,0,Valid


In [18]:
parent_coverage = pd.DataFrame(
    [
        {
            "relationship": "customers with orders",
            "parent_rows_without_child": int(
                (~customers["customer_id"].isin(
                    orders["customer_id"]
                )).sum()
            ),
        },
        {
            "relationship": "orders with order items",
            "parent_rows_without_child": int(
                (~orders["order_id"].isin(
                    order_items["order_id"]
                )).sum()
            ),
        },
        {
            "relationship": "orders with payments",
            "parent_rows_without_child": int(
                (~orders["order_id"].isin(
                    payments["order_id"]
                )).sum()
            ),
        },
        {
            "relationship": "orders with reviews",
            "parent_rows_without_child": int(
                (~orders["order_id"].isin(
                    reviews["order_id"]
                )).sum()
            ),
        },
        {
            "relationship": "products with order items",
            "parent_rows_without_child": int(
                (~products["product_id"].isin(
                    order_items["product_id"]
                )).sum()
            ),
        },
        {
            "relationship": "sellers with order items",
            "parent_rows_without_child": int(
                (~sellers["seller_id"].isin(
                    order_items["seller_id"]
                )).sum()
            ),
        },
    ]
)

display(parent_coverage)

,relationship,parent_rows_without_child
0,customers with orders,0
1,orders with order items,775
2,orders with payments,1
3,orders with reviews,768
4,products with order items,0
5,sellers with order items,0


In [19]:
cardinality_summary = pd.DataFrame(
    [
        {
            "relationship": "orders per customer record",
            "minimum": orders.groupby("customer_id").size().min(),
            "median": orders.groupby("customer_id").size().median(),
            "maximum": orders.groupby("customer_id").size().max(),
        },
        {
            "relationship": "customer records per unique customer",
            "minimum": customers.groupby(
                "customer_unique_id"
            ).size().min(),
            "median": customers.groupby(
                "customer_unique_id"
            ).size().median(),
            "maximum": customers.groupby(
                "customer_unique_id"
            ).size().max(),
        },
        {
            "relationship": "items per order",
            "minimum": order_items.groupby("order_id").size().min(),
            "median": order_items.groupby("order_id").size().median(),
            "maximum": order_items.groupby("order_id").size().max(),
        },
        {
            "relationship": "payments per order",
            "minimum": payments.groupby("order_id").size().min(),
            "median": payments.groupby("order_id").size().median(),
            "maximum": payments.groupby("order_id").size().max(),
        },
        {
            "relationship": "reviews per order",
            "minimum": reviews.groupby("order_id").size().min(),
            "median": reviews.groupby("order_id").size().median(),
            "maximum": reviews.groupby("order_id").size().max(),
        },
    ]
)

display(cardinality_summary)

,relationship,minimum,median,maximum
0,orders per customer record,1,1.00,1
1,customer records per unique customer,1,1.00,17
2,items per order,1,1.00,21
3,payments per order,1,1.00,29
4,reviews per order,1,1.00,3


## 11. Categorical Value Assessment

This process inspects the distinct values and frequency distributions of critical categorical fields. The primary objective is to detect:

- Unexpected categories
- Rare values
- Inconsistent case or spacing
- Missing classifications
- Values requiring business interpretation

In [20]:
categorical_columns = {
    "orders": ["order_status"],
    "payments": ["payment_type"],
    "reviews": ["review_score"],
    "customers": ["customer_state"],
    "sellers": ["seller_state"],
    "geolocation": ["geolocation_state"],
}

In [21]:
for table_name, columns in categorical_columns.items():
    dataframe = datasets[table_name]

    for column in columns:
        print("=" * 30)
        print(f"{table_name}.{column}")
        print("=" * 30)

        display(
            dataframe[column]
            .value_counts(dropna=False)
            .rename("row_count")
            .to_frame()
        )

orders.order_status


,row_count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


payments.payment_type


,row_count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


reviews.review_score


...

,row_count
customer_state,
SP,41746
RJ,12852
MG,11635
RS,5466
PR,5045
SC,3637
BA,3380
DF,2140
ES,2033


sellers.seller_state


,row_count
seller_state,
SP,1849
PR,349
MG,244
SC,190
RJ,171
RS,129
GO,40
DF,30
ES,23


geolocation.geolocation_state


,row_count
geolocation_state,
SP,404268
MG,126336
RJ,121169
RS,61851
PR,57859
SC,38328
BA,36045
GO,20139
ES,16748


In [22]:
valid_brazil_state_codes = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO",
}

state_columns = {
    "customers": "customer_state",
    "sellers": "seller_state",
    "geolocation": "geolocation_state",
}

state_code_checks = []

for table_name, column_name in state_columns.items():
    state_series = (
        datasets[table_name][column_name]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    invalid_mask = (
        state_series.notna()
        & ~state_series.isin(valid_brazil_state_codes)
    )

    state_code_checks.append(
        {
            "table_name": table_name,
            "column_name": column_name,
            "missing_rows": int(state_series.isna().sum()),
            "invalid_state_code_rows": int(
                invalid_mask.sum()
            ),
            "invalid_values": sorted(
                state_series.loc[
                    invalid_mask
                ].dropna().unique().tolist()
            ),
        }
    )

state_code_checks = pd.DataFrame(state_code_checks)

display(state_code_checks)

,table_name,column_name,missing_rows,invalid_state_code_rows,invalid_values
0,customers,customer_state,0,0,[]
1,sellers,seller_state,0,0,[]
2,geolocation,geolocation_state,0,0,[]


## 12. Numerical Data Assessment

Assess important numerical columns for:

- Missing values
- Zero values
- Negative values
- Distribution ranges
- Extreme observations requiring investigation

In [23]:
numerical_columns = {
    "order_items": [
        "order_item_id",
        "price",
        "freight_value",
    ],
    "products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ],
    "payments": [
        "payment_sequential",
        "payment_installments",
        "payment_value",
    ],
    "reviews": [
        "review_score",
    ],
    "geolocation": [
        "geolocation_lat",
        "geolocation_lng",
    ],
}

In [24]:
numerical_summary = []

for table_name, columns in numerical_columns.items():
    dataframe = datasets[table_name]

    for column in columns:
        series = dataframe[column]

        numerical_summary.append(
            {
                "table_name": table_name,
                "column_name": column,
                "count": int(series.count()),
                "missing_count": int(series.isna().sum()),
                "minimum": series.min(),
                "q1": series.quantile(0.25),
                "median": series.median(),
                "q3": series.quantile(0.75),
                "maximum": series.max(),
                "zero_count": int(series.eq(0).sum()),
                "negative_count": int(series.lt(0).sum()),
            }
        )

numerical_summary = pd.DataFrame(numerical_summary)

display(numerical_summary)

,table_name,column_name,count,missing_count,minimum,q1,median,q3,maximum,zero_count,negative_count
0,order_items,order_item_id,112650,0,1.00,1.00,1.00,1.00,21.00,0,0
1,order_items,price,112650,0,0.85,39.90,74.99,134.90,6735.00,0,0
2,order_items,freight_value,112650,0,0.00,13.08,16.26,21.15,409.68,383,0
3,products,product_name_lenght,32341,610,5.00,42.00,51.00,57.00,76.00,0,0
4,products,product_description_lenght,32341,610,4.00,339.00,595.00,972.00,3992.00,0,0
5,products,product_photos_qty,32341,610,1.00,1.00,1.00,3.00,20.00,0,0
6,products,product_weight_g,32949,2,0.00,300.00,700.00,1900.00,40425.00,4,0
7,products,product_length_cm,32949,2,7.00,18.00,25.00,38.00,105.00,0,0
8,products,product_height_cm,32949,2,2.00,8.00,13.00,21.00,105.00,0,0
9,products,product_width_cm,32949,2,6.00,15.00,20.00,30.00,118.00,0,0


In [25]:
numerical_rule_checks = pd.DataFrame(
    [
        {
            "table_name": "order_items",
            "column_name": "price",
            "check": "order item price <= 0",
            "affected_rows": int(
                order_items["price"].le(0).sum()
            ),
        },
        {
            "table_name": "order_items",
            "column_name": "freight_value",
            "check": "freight value < 0",
            "affected_rows": int(
                order_items["freight_value"].lt(0).sum()
            ),
        },
        {
            "table_name": "payments",
            "column_name": "payment_value",
            "check": "payment value <= 0",
            "affected_rows": int(
                payments["payment_value"].le(0).sum()
            ),
        },
        {
            "table_name": "payments",
            "column_name": "payment_installments",
            "check": "payment installments <= 0",
            "affected_rows": int(
                payments["payment_installments"].le(0).sum()
            ),
        },
        {
            "table_name": "products",
            "column_name": "product_weight_g",
            "check": "product weight <= 0",
            "affected_rows": int(
                products["product_weight_g"].le(0).sum()
            ),
        },
        {
            "table_name": "products",
            "column_name": "product_length_cm",
            "check": "product length <= 0",
            "affected_rows": int(
                products["product_length_cm"].le(0).sum()
            ),
        },
        {
            "table_name": "products",
            "column_name": "product_height_cm",
            "check": "product height <= 0",
            "affected_rows": int(
                products["product_height_cm"].le(0).sum()
            ),
        },
        {
            "table_name": "products",
            "column_name": "product_width_cm",
            "check": "product width <= 0",
            "affected_rows": int(
                products["product_width_cm"].le(0).sum()
            ),
        },
        {
            "table_name": "reviews",
            "column_name": "review_score",
            "check": "review score outside 1–5",
            "affected_rows": int(
                (reviews["review_score"].notna() & ~reviews["review_score"].between(1, 5)).sum()),
        }
    ]
)

display(numerical_rule_checks)

,table_name,column_name,check,affected_rows
0,order_items,price,order item price <= 0,0
1,order_items,freight_value,freight value < 0,0
2,payments,payment_value,payment value <= 0,9
3,payments,payment_installments,payment installments <= 0,2
4,products,product_weight_g,product weight <= 0,4
5,products,product_length_cm,product length <= 0,0
6,products,product_height_cm,product height <= 0,0
7,products,product_width_cm,product width <= 0,0
8,reviews,review_score,review score outside 1–5,0


In [26]:
def iqr_outlier_summary(
    dataframe: pd.DataFrame,
    table_name: str,
    column_name: str,
) -> dict:
    """Count values outside the standard 1.5 × IQR range."""

    series = dataframe[column_name].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    flagged_rows = (
        (series < lower_bound)
        | (series > upper_bound)
    ).sum()

    return {
        "table_name": table_name,
        "column_name": column_name,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "flagged_rows": int(flagged_rows),
        "note": "Statistical flag only; not evidence of invalid data",
    }

In [27]:
outlier_columns = [
    ("order_items", "price"),
    ("order_items", "freight_value"),
    ("payments", "payment_value"),
    ("products", "product_weight_g"),
    ("products", "product_length_cm"),
    ("products", "product_height_cm"),
    ("products", "product_width_cm"),
]

outlier_summary = pd.DataFrame(
    [
        iqr_outlier_summary( datasets[table_name], table_name, column_name)
        for table_name, column_name in outlier_columns
    ]
)

display(outlier_summary)

,table_name,column_name,lower_bound,upper_bound,flagged_rows,note
0,order_items,price,-102.60,277.40,8427,Statistical flag only; not evidence of invalid...
1,order_items,freight_value,0.98,33.25,12134,Statistical flag only; not evidence of invalid...
2,payments,payment_value,-115.78,344.41,7981,Statistical flag only; not evidence of invalid...
3,products,product_weight_g,-2100.00,4300.00,4551,Statistical flag only; not evidence of invalid...
4,products,product_length_cm,-12.00,68.00,1380,Statistical flag only; not evidence of invalid...
5,products,product_height_cm,-11.50,40.50,1892,Statistical flag only; not evidence of invalid...
6,products,product_width_cm,-7.50,52.50,912,Statistical flag only; not evidence of invalid...


## 13. Date and Timestamp Assessment

This assessment evaluates whether date fields can be successfully parsed into standard datetime formats and verifies the logical sequence of timestamps across related events.

The audit distinguishes between:

- Parse failures
- Expected missing timestamps
- Invalid Chronological Sequences
- Cases requiring business-process investigation

Temporary parsed copies are used for validation. Raw columns remain unchanged.

In [28]:
orders_dates = orders.copy()
order_items_dates = order_items.copy()
reviews_dates = reviews.copy()

for column in datetime_columns["orders"]:
    orders_dates[column] = pd.to_datetime(
        orders_dates[column],
        errors="coerce"
    )

for column in datetime_columns["order_items"]:
    order_items_dates[column] = pd.to_datetime(
        order_items_dates[column],
        errors="coerce"
    )

for column in datetime_columns["reviews"]:
    reviews_dates[column] = pd.to_datetime(
        reviews_dates[column],
        errors="coerce"
    )

In [29]:
date_range_summary = []

parsed_dateframes = {
    "orders": orders_dates,
    "order_items": order_items_dates,
    "reviews": reviews_dates
}

for table_name, columns in datetime_columns.items():
    dataframe = parsed_dateframes[table_name]

    for column in columns:
        date_range_summary.append(
            {
                "table_name": table_name,
                "column_name": column,
                "minimum_date": dataframe[column].min(),
                "maximum_date": dataframe[column].max(),
                "missing_after_parse": int(dataframe[column].isna().sum())
            }
        )

date_range_summary = pd.DataFrame(date_range_summary)

display(date_range_summary)

,table_name,column_name,minimum_date,maximum_date,missing_after_parse
0,orders,order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0
1,orders,order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160
2,orders,order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783
3,orders,order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965
4,orders,order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0
5,order_items,shipping_limit_date,2016-09-19 00:15:34,2020-04-09 22:35:08,0
6,reviews,review_creation_date,2016-10-02 00:00:00,2018-08-31 00:00:00,0
7,reviews,review_answer_timestamp,2016-10-07 18:32:28,2018-10-29 12:27:35,0


In [30]:
order_date_checks = pd.DataFrame([
    {
        "table_name": "orders",
        "columns_checked": "order_approved_at, order_purchase_timestamp",
        "check": "approval before purchase",
        "affected_rows": int((orders_dates["order_approved_at"].notna() & (orders_dates["order_approved_at"] < orders_dates["order_purchase_timestamp"])).sum())
    },
    {
        "table_name": "orders",
        "columns_checked": "order_delivered_carrier_date, order_purchase_timestamp",
        "check": "carrier handoff before purchase",
        "affected_rows": int((orders_dates["order_delivered_carrier_date"].notna() & (orders_dates["order_delivered_carrier_date"] < orders_dates["order_purchase_timestamp"])).sum())
    },
    {
        "table_name": "orders",
        "columns_checked": "order_delivered_customer_date, order_purchase_timestamp",
        "check": "customer delivery before purchase",
        "affected_rows": int((orders_dates["order_delivered_customer_date"].notna() & (orders_dates["order_delivered_customer_date"] < orders_dates["order_purchase_timestamp"])).sum())
    },
    {
        "table_name": "orders",
        "columns_checked": "order_delivered_customer_date, order_delivered_carrier_date",
        "check": "customer delivery before carrier handoff",
        "affected_rows": int((orders_dates["order_delivered_customer_date"].notna() & orders_dates["order_delivered_carrier_date"].notna() & (orders_dates["order_delivered_customer_date"] < orders_dates["order_delivered_carrier_date"])).sum())
    },
    {
        "table_name": "orders",
        "columns_checked": "order_estimated_delivery_date, order_purchase_timestamp",
        "check": "estimated delivery before purchase",
        "affected_rows": int((orders_dates["order_estimated_delivery_date"] < orders_dates["order_purchase_timestamp"]).sum())
    }
])

display(order_date_checks)


,table_name,columns_checked,check,affected_rows
0,orders,"order_approved_at, order_purchase_timestamp",approval before purchase,0
1,orders,"order_delivered_carrier_date, order_purchase_t...",carrier handoff before purchase,166
2,orders,"order_delivered_customer_date, order_purchase_...",customer delivery before purchase,0
3,orders,"order_delivered_customer_date, order_delivered...",customer delivery before carrier handoff,23
4,orders,"order_estimated_delivery_date, order_purchase_...",estimated delivery before purchase,0


In [31]:
shipping_limit_audit = (
    order_items_dates[
        [
            "order_id",
            "order_item_id",
            "shipping_limit_date",
        ]
    ]
    .merge(
        orders_dates[
            [
                "order_id",
                "order_purchase_timestamp",
            ]
        ],
        on="order_id",
        how="left",
        validate="many_to_one",
    )
)

shipping_limit_before_purchase = int(
    (
        shipping_limit_audit[
            "shipping_limit_date"
        ].notna()
        & (
            shipping_limit_audit[
                "shipping_limit_date"
            ]
            < shipping_limit_audit[
                "order_purchase_timestamp"
            ]
        )
    ).sum()
)

shipping_limit_after_2018 = int(
    (
        shipping_limit_audit[
            "shipping_limit_date"
        ].notna()
        & (
            shipping_limit_audit[
                "shipping_limit_date"
            ]
            >= pd.Timestamp("2019-01-01")
        )
    ).sum()
)

shipping_limit_range_check = pd.DataFrame(
    [
        {
            "table_name": "order_items",
            "columns_checked": "shipping_limit_date",
            "check": (
                "shipping-limit date falls in 2019 or later"
            ),
            "affected_rows": shipping_limit_after_2018,
        }
    ]
)

# This allows the range finding to flow automatically
# into the final audit findings summary.
order_date_checks = pd.concat(
    [
        order_date_checks,
        shipping_limit_range_check,
    ],
    ignore_index=True,
)

print(
    "Shipping-limit dates before purchase:",
    f"{shipping_limit_before_purchase:,}",
)

display(shipping_limit_range_check)

Shipping-limit dates before purchase: 0


,table_name,columns_checked,check,affected_rows
0,order_items,shipping_limit_date,shipping-limit date falls in 2019 or later,4


In [32]:
review_timeline_audit = (
    reviews_dates
    .merge(
        orders_dates[
            [
                "order_id",
                "order_purchase_timestamp",
                "order_delivered_customer_date",
                "order_estimated_delivery_date",
            ]
        ],
        on="order_id",
        how="left",
        validate="many_to_one",
    )
)

# Olist may send the satisfaction survey after actual delivery
# or when the estimated delivery date becomes due.
review_timeline_audit["survey_eligibility_date"] = (
    review_timeline_audit[
        [
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ]
    ]
    .min(axis=1, skipna=True)
    .dt.normalize()
)

review_timeline_audit["survey_sent_date"] = (
    review_timeline_audit["review_creation_date"]
    .dt.normalize()
)

review_timeline_audit["purchase_date"] = (
    review_timeline_audit["order_purchase_timestamp"]
    .dt.normalize()
)

In [33]:
review_date_checks = pd.DataFrame(
    [
        {
            "table_name": "reviews",
            "columns_checked": (
                "review_creation_date, "
                "order_purchase_timestamp"
            ),
            "check": "survey sent before purchase date",
            "affected_rows": int(
                (
                    review_timeline_audit[
                        "survey_sent_date"
                    ].notna()
                    & review_timeline_audit[
                        "purchase_date"
                    ].notna()
                    & (
                        review_timeline_audit[
                            "survey_sent_date"
                        ]
                        < review_timeline_audit[
                            "purchase_date"
                        ]
                    )
                ).sum()
            ),
        },
        {
            "table_name": "reviews",
            "columns_checked": (
                "review_answer_timestamp, "
                "review_creation_date"
            ),
            "check": "survey response before survey was sent",
            "affected_rows": int(
                (
                    review_timeline_audit[
                        "review_answer_timestamp"
                    ].notna()
                    & review_timeline_audit[
                        "review_creation_date"
                    ].notna()
                    & (
                        review_timeline_audit[
                            "review_answer_timestamp"
                        ]
                        < review_timeline_audit[
                            "review_creation_date"
                        ]
                    )
                ).sum()
            ),
        },
        {
            "table_name": "reviews",
            "columns_checked": (
                "review_creation_date, "
                "survey_eligibility_date"
            ),
            "check": (
                "survey sent before actual-delivery "
                "or estimated-due eligibility date"
            ),
            "affected_rows": int(
                (
                    review_timeline_audit[
                        "survey_sent_date"
                    ].notna()
                    & review_timeline_audit[
                        "survey_eligibility_date"
                    ].notna()
                    & (
                        review_timeline_audit[
                            "survey_sent_date"
                        ]
                        < review_timeline_audit[
                            "survey_eligibility_date"
                        ]
                    )
                ).sum()
            ),
        },
    ]
)

display(review_date_checks)

,table_name,columns_checked,check,affected_rows
0,reviews,"review_creation_date, order_purchase_timestamp",survey sent before purchase date,64
1,reviews,"review_answer_timestamp, review_creation_date",survey response before survey was sent,0
2,reviews,"review_creation_date, survey_eligibility_date",survey sent before actual-delivery or estimate...,606


### Interpretation Note :

- `review_creation_date` represents the date on which the satisfaction survey was sent, while `review_answer_timestamp` records when the survey was answered.

- The survey may be sent after actual delivery or when the estimated delivery date becomes due. Therefore, a survey date earlier than the recorded delivery timestamp is not automatically an anomaly.

- The audit compares calendar dates rather than exact timestamps because `review_creation_date` is recorded at midnight. This prevents same-day survey and delivery records from being incorrectly classified as chronological violations.

## 14. Text Field and Lookup Assessment

This section evaluates issues, including:

- Leading or trailing whitespace
- Empty strings
- Case variation
- Numeric-only city names
- Unexpected lengths

##### Accents and special characters are not treated as errors because Portuguese names and review text legitimately contain them.

In [34]:
text_audit_rows = []

for table_name, dataframe in datasets.items():
    text_columns = dataframe.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:
        series = dataframe[column].astype("string")

        text_audit_rows.append(
            {
                "table_name": table_name,
                "column_name": column,
                "missing_count": int(series.isna().sum()),
                "empty_string_count": int(
                    series.eq("").fillna(False).sum()
                ),
                "whitespace_only_count": int(
                    series.str.fullmatch(r"\s+")
                    .fillna(False)
                    .sum()
                ),
                "leading_or_trailing_space_count": int(
                    series.ne(series.str.strip())
                    .fillna(False)
                    .sum()
                ),
                "minimum_length": (
                    series.dropna().str.len().min()
                    if series.notna().any()
                    else np.nan
                ),
                "maximum_length": (
                    series.dropna().str.len().max()
                    if series.notna().any()
                    else np.nan
                ),
            }
        )

text_field_audit = pd.DataFrame(text_audit_rows)

display(
    text_field_audit.sort_values(
        [
            "leading_or_trailing_space_count",
            "empty_string_count",
        ],
        ascending=False,
    )
)

,table_name,column_name,missing_count,empty_string_count,whitespace_only_count,leading_or_trailing_space_count,minimum_length,maximum_length
21,reviews,review_comment_message,58247,0,27,9451,1,208
20,reviews,review_comment_title,87656,0,2,1998,1,26
29,geolocation,geolocation_city,0,0,0,1,2,38
0,customers,customer_id,0,0,0,0,32,32
1,customers,customer_unique_id,0,0,0,0,32,32
2,customers,customer_city,0,0,0,0,3,32
3,customers,customer_state,0,0,0,0,2,2
4,orders,order_id,0,0,0,0,32,32
5,orders,customer_id,0,0,0,0,32,32
6,orders,order_status,0,0,0,0,7,11


In [35]:
city_columns = {
    "customers": "customer_city",
    "sellers": "seller_city",
    "geolocation": "geolocation_city",
}

numeric_only_city_checks = []

for table_name, column_name in city_columns.items():
    series = datasets[table_name][column_name].astype("string")

    numeric_only_city_checks.append(
        {
            "table_name": table_name,
            "column_name": column_name,
            "numeric_only_rows": int(
                series.str.fullmatch(r"\d+")
                .fillna(False)
                .sum()
            ),
        }
    )

numeric_only_city_checks = pd.DataFrame(
    numeric_only_city_checks
)

display(numeric_only_city_checks)

,table_name,column_name,numeric_only_rows
0,customers,customer_city,0
1,sellers,seller_city,1
2,geolocation,geolocation_city,0


In [36]:
product_categories_without_translation = (
    products.loc[
        products["product_category_name"].notna(),
        ["product_category_name"],
    ]
    .drop_duplicates()
    .merge(
        category_translation[
            ["product_category_name"]
        ],
        on="product_category_name",
        how="left",
        indicator=True,
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)

translations_without_products = (
    category_translation[
        ["product_category_name"]
    ]
    .merge(
        products.loc[
            products["product_category_name"].notna(),
            ["product_category_name"],
        ].drop_duplicates(),
        on="product_category_name",
        how="left",
        indicator=True,
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)

print(
    "Product categories without translation:",
    len(product_categories_without_translation),
)

print(
    "Translations not used by products:",
    len(translations_without_products),
)

display(product_categories_without_translation)
display(translations_without_products)

Product categories without translation: 2
Translations not used by products: 0


,product_category_name
63,pc_gamer
69,portateis_cozinha_e_preparadores_de_alimentos


,product_category_name


## 15. Audit Findings Summary

The raw-data audit has assessed the dataset's structure, grain, key integrity, missingness, duplication, relationships, value domains, timestamps, and text quality. The purpose of this summary is to identify which findings should be carried forward for investigation in `C2_data_cleaning.ipynb`.

A finding is not automatically a cleaning requirement. Each issue must first be classified as:

- Expected business behaviour
- Optional or conditionally applicable data
- Formatting inconsistency
- Data type issue
- Potential business-rule violation
- Referential integrity issue
- Unresolved data-quality concern

In [37]:
audit_findings = []


def add_finding(
    audit_area: str,
    table_name: str,
    column_name: str,
    finding: str,
    requires_investigation: str = "Yes",
) -> None:
    audit_findings.append(
        {
            "audit_area": audit_area,
            "table_name": table_name,
            "column_name": column_name,
            "finding": finding,
            "requires_investigation": requires_investigation,
        }
    )


# Missing values
for _, row in missing_value_summary.iterrows():
    add_finding(
        audit_area="Missing values",
        table_name=row["table_name"],
        column_name=row["column_name"],
        finding=(
            f"{int(row['missing_count']):,} missing values "
            f"({row['missing_pct']:.2f}%)"
        ),
    )


# Exact duplicate rows
for _, row in duplicate_summary.iterrows():
    if row["rows_in_duplicate_groups"] > 0:
        add_finding(
            audit_area="Duplicates",
            table_name=row["table_name"],
            column_name="All columns",
            finding=(
                f"{int(row['rows_in_duplicate_groups']):,} rows belong "
                f"to {int(row['duplicate_groups']):,} exact duplicate groups; "
                f"{int(row['excess_duplicate_rows']):,} rows exceed one "
                "retained copy per group"
            ),
        )


# Invalid or incomplete proposed keys
for _, row in key_validation.iterrows():
    if not row["valid_key"]:
        add_finding(
            audit_area="Key integrity",
            table_name=row["table_name"],
            column_name=row["key_columns"],
            finding=(
                f"Unique: {row['is_unique']}; "
                f"complete: {row['is_complete']}"
            ),
        )


# Repeated review identifiers
duplicated_review_id_rows = int(
    reviews["review_id"].duplicated(keep=False).sum()
)

if duplicated_review_id_rows > 0:
    add_finding(
        audit_area="Key interpretation",
        table_name="reviews",
        column_name="review_id",
        finding=(
            f"{duplicated_review_id_rows:,} rows contain a review_id "
            "that appears more than once; the validated composite key is "
            "review_id + order_id"
        ),
    )


# Orphan child rows
for _, row in referential_integrity.iterrows():
    if row["orphan_rows"] > 0:
        add_finding(
            audit_area="Referential integrity",
            table_name=row["child_table"],
            column_name=row["child_key"],
            finding=(
                f"{int(row['orphan_rows']):,} orphan rows in "
                f"{row['relationship']}"
            ),
        )


# Parent records without child records
for _, row in parent_coverage.iterrows():
    if row["parent_rows_without_child"] > 0:
        add_finding(
            audit_area="Relationship coverage",
            table_name="Multiple",
            column_name="See relationship",
            finding=(
                f"{row['relationship']}: "
                f"{int(row['parent_rows_without_child']):,} parent rows "
                "have no matching child record"
            ),
        )


# Datetime conversion requirements
for _, row in datetime_type_audit.iterrows():
    if row["requires_conversion"] or row["parse_failures"] > 0:
        add_finding(
            audit_area="Data types",
            table_name=row["table_name"],
            column_name=row["column_name"],
            finding=(
                f"Current dtype: {row['current_dtype']}; "
                f"parse failures: {int(row['parse_failures']):,}"
            ),
        )


# Numerical rule exceptions
for _, row in numerical_rule_checks.iterrows():
    if row["affected_rows"] > 0:
        add_finding(
            audit_area="Numerical validation",
            table_name=row["table_name"],
            column_name=row["column_name"],
            finding=(
                f"{row['check']}: "
                f"{int(row['affected_rows']):,} rows"
            ),
        )


# Statistical outlier flags
for _, row in outlier_summary.iterrows():
    if row["flagged_rows"] > 0:
        add_finding(
            audit_area="Statistical review",
            table_name=row["table_name"],
            column_name=row["column_name"],
            finding=(
                f"{int(row['flagged_rows']):,} rows fall outside the "
                "1.5 × IQR range; this is a review flag, not evidence "
                "of invalid data"
            ),
        )


# Date-rule exceptions
date_checks_for_summary = pd.concat(
    [
        order_date_checks,
        review_date_checks,
    ],
    ignore_index=True,
)

for _, row in date_checks_for_summary.iterrows():
    if row["affected_rows"] > 0:
        add_finding(
            audit_area="Date consistency",
            table_name=row["table_name"],
            column_name=row["columns_checked"],
            finding=(
                f"{row['check']}: "
                f"{int(row['affected_rows']):,} rows"
            ),
        )


if shipping_limit_before_purchase > 0:
    add_finding(
        audit_area="Date consistency",
        table_name="order_items",
        column_name="shipping_limit_date",
        finding=(
            f"{int(shipping_limit_before_purchase):,} shipping-limit "
            "dates occur before purchase"
        ),
    )


# Text-quality findings
for _, row in text_field_audit.iterrows():
    issue_count = (
        row["empty_string_count"]
        + row["whitespace_only_count"]
        + row["leading_or_trailing_space_count"]
    )

    if issue_count > 0:
        add_finding(
            audit_area="Text quality",
            table_name=row["table_name"],
            column_name=row["column_name"],
            finding=(
                f"Empty strings: {int(row['empty_string_count']):,}; "
                f"whitespace-only values: "
                f"{int(row['whitespace_only_count']):,}; "
                f"leading/trailing spaces: "
                f"{int(row['leading_or_trailing_space_count']):,}"
            ),
        )


# Numeric-only city values
for _, row in numeric_only_city_checks.iterrows():
    if row["numeric_only_rows"] > 0:
        add_finding(
            audit_area="Text quality",
            table_name=row["table_name"],
            column_name=row["column_name"],
            finding=(
                f"{int(row['numeric_only_rows']):,} numeric-only city value"
            ),
        )


# Unmapped product categories
if len(product_categories_without_translation) > 0:
    add_finding(
        audit_area="Lookup coverage",
        table_name="products",
        column_name="product_category_name",
        finding=(
            f"{len(product_categories_without_translation):,} product "
            "categories have no English translation"
        ),
    )


# Unexpected categorical values
not_defined_payments = int(
    payments["payment_type"].eq("not_defined").sum()
)

if not_defined_payments > 0:
    add_finding(
        audit_area="Categorical validation",
        table_name="payments",
        column_name="payment_type",
        finding=(
            f"{not_defined_payments:,} rows use the category "
            "'not_defined'"
        ),
    )


audit_findings_summary = (
    pd.DataFrame(audit_findings)
    .sort_values(
        ["audit_area", "table_name", "column_name"]
    )
    .reset_index(drop=True)
)

display(audit_findings_summary)

,audit_area,table_name,column_name,finding,requires_investigation
0,Categorical validation,payments,payment_type,3 rows use the category 'not_defined',Yes
1,Data types,order_items,shipping_limit_date,Current dtype: object; parse failures: 0,Yes
2,Data types,orders,order_approved_at,Current dtype: object; parse failures: 0,Yes
3,Data types,orders,order_delivered_carrier_date,Current dtype: object; parse failures: 0,Yes
4,Data types,orders,order_delivered_customer_date,Current dtype: object; parse failures: 0,Yes
5,Data types,orders,order_estimated_delivery_date,Current dtype: object; parse failures: 0,Yes
6,Data types,orders,order_purchase_timestamp,Current dtype: object; parse failures: 0,Yes
7,Data types,reviews,review_answer_timestamp,Current dtype: object; parse failures: 0,Yes
8,Data types,reviews,review_creation_date,Current dtype: object; parse failures: 0,Yes
9,Date consistency,order_items,shipping_limit_date,shipping-limit date falls in 2019 or later: 4 ...,Yes


## Final Audit Conclusion

The audit establishes a baseline for all nine raw Olist datasets. Before database loading, the cleaning stage must investigate or document the following areas:

- Timestamp columns currently stored as text require controlled conversion.
- Missing values must be interpreted according to their business context rather than handled uniformly.
- Exact duplicate records require table-specific investigation.
- Composite keys must be preserved where individual identifiers are not unique.
- Category-translation and geolocation joins require careful treatment because they are logical lookup relationships rather than conventional one-to-many foreign-key relationships.
- Numerical and temporal anomalies require investigation before any correction or exclusion.
- Shipping-limit dates outside the documented dataset period require targeted validation.
- Statistical outliers must not be removed without evidence that they represent invalid records.
- Review and geographic text must retain legitimate Portuguese characters and accents.
- Review dates must be interpreted using the satisfaction-survey workflow documented for the source dataset.

### No cleaning actions were applied in this notebook.

The next notebook, `C2_data_cleaning.ipynb`, will:

1. Review each audit finding.
2. Investigate its business context.
3. Classify it as valid behaviour, a formatting issue, or a data-quality problem.
4. Apply only justified cleaning transformations.
5. Document the effect of every change.